# Synthea Claims Analytics — Source Audit

**Platform:** Microsoft Fabric Lakehouse  
**Purpose:** Profile the raw Synthea claims sources before Bronze-to-Silver transformation.  
**Data classification:** Synthetic healthcare data; no real PHI.

This notebook validates table grain, key uniqueness, missingness, referential integrity, date logic, and type-specific financial fields. It does not alter the source files.

## Confirmed source design

| Source | Expected grain | Rows in supplied extract | Candidate key |
|---|---|---:|---|
| patients.csv | One row per patient | 1,163 | Id |
| encounters.csv | One row per encounter | 61,459 | Id |
| claims.csv | One row per claim header | 117,889 | Id |
| claims_transactions.csv | One row per claim transaction event | 711,238 | ID |
| payers.csv | One row per payer | 10 | Id |
| payer_transitions.csv | One row per coverage interval | 53,101 | Composite key to be validated |

Important: `PATIENTINSURANCEID` in the transaction source behaves like the patient identifier in this extract. It must not be joined to `payers.Id` or treated as a member-policy key without validation.

In [1]:
from functools import reduce
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

BASE_PATH = "Files/bronze/synthea"

FILES = {
    "patients": "patients.csv",
    "encounters": "encounters.csv",
    "claims": "claims.csv",
    "claims_transactions": "claims_transactions.csv",
    "payers": "payers.csv",
    "payer_transitions": "payer_transitions.csv",
}

EXPECTED_KEYS = {
    "patients": ["Id"],
    "encounters": ["Id"],
    "claims": ["Id"],
    "claims_transactions": ["ID"],
    "payers": ["Id"],
    "payer_transitions": ["PATIENT", "START_YEAR", "END_YEAR", "PAYER", "SECONDARY_PAYER"],
}

StatementMeta(, 64b0a6ce-29e8-43eb-a6bb-80ee8fc6b65f, 3, Finished, Available, Finished, False)

## 1. Read raw CSV files

All columns are initially read as strings. Explicit data types will be enforced in the Silver transformation notebook after the source audit is approved.

In [2]:
dfs = {}
for table_name, file_name in FILES.items():
    path = f"{BASE_PATH}/{file_name}"
    dfs[table_name] = (
        spark.read
        .option("header", True)
        .option("inferSchema", False)
        .option("mode", "PERMISSIVE")
        .csv(path)
    )

inventory_rows = []
for table_name, df in dfs.items():
    inventory_rows.append((
        table_name, FILES[table_name], df.count(), len(df.columns),
        ", ".join(EXPECTED_KEYS[table_name])
    ))

inventory_df = spark.createDataFrame(
    inventory_rows,
    ["table_name", "source_file", "row_count", "column_count", "candidate_key"]
)
display(inventory_df.orderBy("table_name"))

StatementMeta(, 64b0a6ce-29e8-43eb-a6bb-80ee8fc6b65f, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bbadbe86-63d9-4cd4-8568-cc17ab56da44)

## 2. Column-level completeness

Blank strings and nulls are both counted as missing. High missingness is not automatically an error; optional secondary-insurance and diagnosis fields are expected to be sparse.

In [3]:
missing_frames = []
for table_name, df in dfs.items():
    row_count = df.count()
    expressions = [
        F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c)
        for c in df.columns
    ]
    counts = df.agg(*expressions).first().asDict()
    rows = [
        (table_name, c, int(counts[c]), float(counts[c] / row_count) if row_count else None)
        for c in df.columns
    ]
    missing_frames.append(spark.createDataFrame(rows, ["table_name", "column_name", "missing_count", "missing_pct"]))

missingness_df = reduce(DataFrame.unionByName, missing_frames)
display(missingness_df.orderBy(F.desc("missing_pct"), "table_name", "column_name"))

StatementMeta(, 64b0a6ce-29e8-43eb-a6bb-80ee8fc6b65f, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 25e823db-b3be-4203-a8dc-d02d69eac4a8)

## 3. Key uniqueness and exact duplicates

In [4]:
key_results = []
for table_name, df in dfs.items():
    keys = EXPECTED_KEYS[table_name]
    total = df.count()
    distinct_key_count = df.select(*keys).distinct().count()
    missing_key_rows = df.filter(
        reduce(lambda a, b: a | b, [F.col(c).isNull() | (F.trim(F.col(c)) == "") for c in keys])
    ).count()
    duplicate_key_rows = (
        df.groupBy(*keys).count().filter(F.col("count") > 1)
        .agg(F.coalesce(F.sum("count"), F.lit(0)).alias("duplicate_rows"))
        .first()["duplicate_rows"]
    )
    exact_duplicate_rows = total - df.distinct().count()
    key_results.append((table_name, total, distinct_key_count, missing_key_rows, int(duplicate_key_rows), exact_duplicate_rows))

key_audit_df = spark.createDataFrame(
    key_results,
    ["table_name", "row_count", "distinct_candidate_keys", "missing_key_rows", "rows_in_duplicate_key_groups", "exact_duplicate_rows"]
)
display(key_audit_df.orderBy("table_name"))

StatementMeta(, 64b0a6ce-29e8-43eb-a6bb-80ee8fc6b65f, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7cafc173-7733-4b05-ae2d-913152fb7561)

## 4. Referential-integrity checks

Each result counts child rows whose nonblank key does not exist in the expected parent table. A zero count passes the check.

In [5]:
def orphan_count(child_df, child_col, parent_df, parent_col, zero_is_missing=False):
    child = child_df.select(F.trim(F.col(child_col)).alias("child_key"))
    child = child.filter(F.col("child_key").isNotNull() & (F.col("child_key") != ""))
    if zero_is_missing:
        child = child.filter(F.col("child_key") != "0")
    parent = parent_df.select(F.trim(F.col(parent_col)).alias("parent_key")).distinct()
    return child.join(parent, child.child_key == parent.parent_key, "left_anti").count()

relationship_rules = [
    ("claims.PATIENTID -> patients.Id", "claims", "PATIENTID", "patients", "Id", False),
    ("claims.APPOINTMENTID -> encounters.Id", "claims", "APPOINTMENTID", "encounters", "Id", False),
    ("claims.PRIMARYPATIENTINSURANCEID -> payers.Id", "claims", "PRIMARYPATIENTINSURANCEID", "payers", "Id", True),
    ("claims.SECONDARYPATIENTINSURANCEID -> payers.Id", "claims", "SECONDARYPATIENTINSURANCEID", "payers", "Id", True),
    ("claims_transactions.CLAIMID -> claims.Id", "claims_transactions", "CLAIMID", "claims", "Id", False),
    ("claims_transactions.PATIENTID -> patients.Id", "claims_transactions", "PATIENTID", "patients", "Id", False),
    ("claims_transactions.APPOINTMENTID -> encounters.Id", "claims_transactions", "APPOINTMENTID", "encounters", "Id", False),
    ("encounters.PATIENT -> patients.Id", "encounters", "PATIENT", "patients", "Id", False),
    ("encounters.PAYER -> payers.Id", "encounters", "PAYER", "payers", "Id", True),
    ("payer_transitions.PATIENT -> patients.Id", "payer_transitions", "PATIENT", "patients", "Id", False),
    ("payer_transitions.PAYER -> payers.Id", "payer_transitions", "PAYER", "payers", "Id", True),
    ("payer_transitions.SECONDARY_PAYER -> payers.Id", "payer_transitions", "SECONDARY_PAYER", "payers", "Id", True),
]

ri_rows = []
for label, child_name, child_col, parent_name, parent_col, zero_is_missing in relationship_rules:
    orphans = orphan_count(dfs[child_name], child_col, dfs[parent_name], parent_col, zero_is_missing)
    ri_rows.append((label, orphans, "PASS" if orphans == 0 else "REVIEW"))

referential_integrity_df = spark.createDataFrame(ri_rows, ["relationship", "orphan_rows", "status"])
display(referential_integrity_df)

StatementMeta(, 64b0a6ce-29e8-43eb-a6bb-80ee8fc6b65f, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 27388b5d-e655-4fbb-8359-6d5042f116ae)

## 5. Claims cardinality

`APPOINTMENTID` is not a claim key. One encounter may generate multiple claim headers. Each claim may contain multiple transaction events.

In [6]:
claims_per_encounter = dfs["claims"].groupBy("APPOINTMENTID").count()
transactions_per_claim = dfs["claims_transactions"].groupBy("CLAIMID").count()

cardinality_df = spark.createDataFrame([
    ("Claims per encounter", float(claims_per_encounter.agg(F.min("count")).first()[0]), float(claims_per_encounter.approxQuantile("count", [0.5], 0.001)[0]), float(claims_per_encounter.agg(F.avg("count")).first()[0]), float(claims_per_encounter.agg(F.max("count")).first()[0])),
    ("Transactions per claim", float(transactions_per_claim.agg(F.min("count")).first()[0]), float(transactions_per_claim.approxQuantile("count", [0.5], 0.001)[0]), float(transactions_per_claim.agg(F.avg("count")).first()[0]), float(transactions_per_claim.agg(F.max("count")).first()[0])),
], ["measure", "minimum", "median", "mean", "maximum"])
display(cardinality_df)

StatementMeta(, 64b0a6ce-29e8-43eb-a6bb-80ee8fc6b65f, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bc8aef40-c621-4915-8b02-85d9b1979f2d)

## 6. Transaction-type financial mapping

Do not calculate all transaction values from `AMOUNT`. In this source:

- CHARGE uses `AMOUNT` and `UNITAMOUNT`.
- PAYMENT uses `PAYMENTS`.
- TRANSFERIN and TRANSFEROUT use `TRANSFERS`; TRANSFERIN also carries `AMOUNT`.
- `ADJUSTMENTS` is currently unpopulated, so an adjustment KPI is unsupported in this extract.

In [7]:
tx = dfs["claims_transactions"]
money_cols = ["AMOUNT", "UNITAMOUNT", "PAYMENTS", "ADJUSTMENTS", "TRANSFERS", "OUTSTANDING"]
typed_tx = tx
for c in money_cols:
    typed_tx = typed_tx.withColumn(c, F.col(c).cast(DecimalType(18, 2)))

financial_expressions = []
for c in money_cols:
    financial_expressions.extend([
        F.count(c).alias(f"{c}_nonmissing"),
        F.sum(c).alias(f"{c}_sum"),
    ])

transaction_financial_profile_df = typed_tx.groupBy("TYPE").agg(*financial_expressions).orderBy("TYPE")
display(transaction_financial_profile_df)

StatementMeta(, 64b0a6ce-29e8-43eb-a6bb-80ee8fc6b65f, 9, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 96ebe051-78c5-4ce4-bdc4-f4b180c8ecff)

## 7. Source-field anomaly: PATIENTINSURANCEID

In [8]:
insurance_field_check_df = typed_tx.agg(
    F.count("*").alias("transaction_rows"),
    F.sum(F.when(F.col("PATIENTINSURANCEID").isNull() | (F.trim(F.col("PATIENTINSURANCEID")) == ""), 1).otherwise(0)).alias("blank_rows"),
    F.sum(F.when(F.col("PATIENTINSURANCEID") == F.col("PATIENTID"), 1).otherwise(0)).alias("equals_patient_id_rows"),
    F.sum(F.when((F.col("PATIENTINSURANCEID").isNotNull()) & (F.col("PATIENTINSURANCEID") != "") & (F.col("PATIENTINSURANCEID") != "0") & (F.col("PATIENTINSURANCEID") != F.col("PATIENTID")), 1).otherwise(0)).alias("nonblank_mismatch_rows"),
)
display(insurance_field_check_df)

StatementMeta(, 64b0a6ce-29e8-43eb-a6bb-80ee8fc6b65f, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 53ceb19d-93b4-446a-8c88-7a71d67a212c)

## 8. Temporal validation

In [9]:
encounter_dates = (dfs["encounters"]
    .withColumn("start_ts", F.to_timestamp("START"))
    .withColumn("stop_ts", F.to_timestamp("STOP")))
coverage_dates = (dfs["payer_transitions"]
    .withColumn("coverage_start_ts", F.to_timestamp("START_YEAR"))
    .withColumn("coverage_end_ts", F.to_timestamp("END_YEAR")))

temporal_checks_df = spark.createDataFrame([
    ("Encounter STOP before START", encounter_dates.filter(F.col("stop_ts") < F.col("start_ts")).count()),
    ("Coverage END_YEAR before START_YEAR", coverage_dates.filter(F.col("coverage_end_ts") < F.col("coverage_start_ts")).count()),
], ["rule", "failed_rows"]).withColumn("status", F.when(F.col("failed_rows") == 0, "PASS").otherwise("REVIEW"))
display(temporal_checks_df)

StatementMeta(, 64b0a6ce-29e8-43eb-a6bb-80ee8fc6b65f, 11, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 5d0b9dd8-4848-4887-8fec-037ee65b954a)

## 9. Audit interpretation

Expected results for the supplied extract:

- No exact duplicate rows in the six source tables.
- `Id`/`ID` is unique in patients, encounters, claims, claim transactions, and payers.
- Every claim has at least one transaction; every encounter has at least one claim header.
- Patient, encounter, claim, and payer foreign-key checks pass for the intended relationships.
- No encounter ends before it starts, and no coverage interval ends before it starts.
- `claims.APPOINTMENTID` is not unique and must not be used as the claim primary key.
- `claims_transactions.PATIENTINSURANCEID` must remain a documented source anomaly, not a payer relationship.
- PAYMENT values must come from `PAYMENTS`, not `AMOUNT`.

### Next notebook

Create explicit schemas, standardize nulls, cast dates and decimals, add ingestion metadata, derive a normalized transaction amount, quarantine failed records, and write the Silver Delta tables.